In [1]:
# 한국어를 이용한 감성 분류
# 이진분류 문제 => 리뷰 내용이 긍정인지 부정인지 학습한 후 예측!
# 기존에 배웠던 Token, Tokenizer, Padding, vacabulary, One-hot, Embedding
# 이런 개념들이 어떻게 사용되는지 코드로 확인해 보면 될 듯 해요!

In [2]:
%reset -f

In [10]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime
import tensorflow as tf

2025-07-09 11:11:29.164232: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-09 11:11:29.302912: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-07-09 11:11:29.372702: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-07-09 11:11:29.373102: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-09 11:11:29.479458: I tensorflow/core/platform/cpu_feature_gua

In [11]:
# 네이버에서 제공하는 네이버 영화 리뷰 데이터셋을 다운로드 해요
train_file = tf.keras.utils.get_file(
    cache_dir = './data',  # 다운로드 경로
    fname = 'ratings_train.txt', # 파일명
    origin = 'https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt',  # 다운로드 받는 파일의 경로
    extract = True
) 

/tmp/ipykernel_3898/3950924462.py:2: UserWarning: Could not extract archive.
  train_file = tf.keras.utils.get_file(


In [12]:
train_path = os.path.join('./data/datasets', 'ratings_train.txt')
train_df = pd.read_csv(train_path, sep='\t')    # \t 는 탭 표시
display(train_df.head())
print(train_df.shape) # => 리뷰 개수가 15만개에요

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


(150000, 3)


In [13]:
test_file = tf.keras.utils.get_file(
    cache_dir = './data',  # 다운로드 경로
    fname = 'ratings_test.txt', # 파일명
    origin = 'https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt',  # 다운로드 받는 파일의 경로
    extract = True
) 
test_path = os.path.join('./data/datasets', 'ratings_test.txt')
test_df = pd.read_csv(test_path, sep='\t') 
display(test_df.head())
print(test_df.shape) # => test용 리뷰 데이터는 5만개에요

/tmp/ipykernel_3898/2268427059.py:1: UserWarning: Could not extract archive.
  test_file = tf.keras.utils.get_file(


,id,document,label
0,6270596,굳 ㅋ,1
1,9274899,GDNTOPCLASSINTHECLUB,0
2,8544678,뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아,0
3,6825595,지루하지는 않은데 완전 막장임... 돈주고 보기에는....,0
4,6723715,3D만 아니었어도 별 다섯 개 줬을텐데.. 왜 3D로 나와서 제 심기를 불편하게 하죠??,0


(50000, 3)


In [14]:
# 간단하게 데이터 특성에 대해서 알아보아요 (전처리 텍스트라 이상치는 없다고봐야됨)
# 이진분류하고 있기 떄문에 적어도 긍정과 부정이 1:1 비율로 있어야 학습이 잘 이루어질 것 같아요
cnt = train_df['label'].value_counts()  # 변수안에 있는 유니크 밸류를 알려주는 함수
print(cnt) # 보니 거진 1:1 비율임. 데이터 뷸균형 문제는 없을것 같아요 

label
0    75173
1    74827
Name: count, dtype: int64


In [15]:
# 결측치 확인해봐요
print(train_df.isnull().sum()) 
# 결측치가 5개가 있어요. 나중에 삭제를 해야 할 것 같아요. 트레인 데이터가 15만건이고 이중 5개만 결측치니까. 그리고 자연어가 null인거라 삭제가 좋음

id          0
document    5
label       0
dtype: int64


In [ ]:
# Token을 구별해서 이 token으로 vacabulary를 구성해야 해요!
# 그런데 한국어는 이 토큰을 분리해 내는 작업자체가 어려워요! -> 형태소 분석으로
# 형태소 분석은 어떻게 하면 되나요?
# 라이브러리가 있어요! => 여러개가 있어요! (여러가지 방법이 있어요)
# 그럼 가장 많이 사용하는 형태소 분석기는 어떤것들이 있나요?
# 1. mecab(메캅) - 가장 널리 사용되요 C++기반 빠르고 정확해요. 실무에서 주로 사용. 설치가 약간 복잡
# 2. Okt(Open koran text processor) - 학습용으로 사용되요. 설치가 간단
# 3. Kiwi - 딥러닝 기반으로 띄어쓰기 오류에 강해요

# 우리는 메캅을 설치해서 사용할 거에요
# 우븐트 리눅스에 설치하는 과정

# sudo apt update
# sudo apt install -y make curl git build-essential autoconf automake libtool
# Mecab 본체 설치
# sudo apt install -y mecab libmecab-dev mecab-ipadic-utf8

# Mecab 한국어 사전 설치 Token을 구별해서 이 token으로 vacabulary를 구성해야 해요!
# 그런데 한국어는 이 토큰을 분리해 내는 작업자체가 어려워요! -> 형태소 분석으로
# 형태소 분석은 어떻게 하면 되나요?
# 라이브러리가 있어요! => 여러개가 있어요! (여러가지 방법이 있어요)
# 그럼 가장 많이 사용하는 형태소 분석기는 어떤것들이 있나요?
# 1. mecab(메캅) - 가장 널리 사용되요 C++기반 빠르고 정확해요. 실무에서 주로 사용. 설치가 약간 복잡
# 2. Okt(Open koran text processor) - 학습용으로 사용되요. 설치가 간단
# 3. Kiwi - 딥러닝 기반으로 띄어쓰기 오류에 강해요

# 우리는 메캅을 설치해서 사용할 거에요
# 우븐트 리눅스에 설치하는 과정

# sudo apt update
# sudo apt install -y make curl git build-essential autoconf automake libtool
# Mecab 본체 설치
# sudo apt install -y mecab libmecab-dev mecab-ipadic-utf8

# Mecab 한국어 사전 설치
# cd /tmp

# # mecab-ko-dic 다운로드
# wget https://bitbucket.org/eunjeon/mecab-ko-dic/downloads/mecab-ko-dic-2.1.1-20180720.tar.gz
# tar zxfv mecab-ko-dic-2.1.1-20180720.tar.gz
# cd mecab-ko-dic-2.1.1-20180720

# 설치 경로 확인
# mecab-config --dicdir    -> 내 경로는 이거임 /usr/lib/x86_64-linux-gnu/mecab/dic

In [ ]:
# 메캅 설치가 다 됬으면 확인을 한 후에
# 이 메캅을 파이썬에서 사용할 거에요
# 이 작업을 수행해 줄 파이썬 모둘을 설치해야 해요
# pip install konlay

In [20]:
# 코드상에서 형태소 분석을 해 보아요
from konlpy.tag import Mecab, Okt

okt = Okt()  # 형태소 분석을 할 수 있는 객체를 하나 만들고
# mecab = Mecab(dicpath='/usr/lib/x86_64-linux-gnu/mecab/dic/mecab-ko-dic')
mecab = Mecab()

text = "이것은 소리없는 아우성. 저 푸른 해원을 향하여 흔드는 노스텔지어의 손수건"
print(okt.morphs(text))

['이', '것', '은', '소리', '없는', '아우성', '.', '저', '푸른', '해원', '을', '향', '하여', '흔드는', '노스', '텔', '지', '어의', '손수건']


In [4]:
import MeCab
mecab = MeCab.Tagger("")  # now works without specifying -d
print(mecab.parse("형태소 분석이 이제 정상적으로 됩니다."))


형태소	NNG,*,F,형태소,Compound,*,*,형태/NNG/*+소/NNG/*
분석	NNG,행위,T,분석,*,*,*,*
이	JKS,*,F,이,*,*,*,*
이제	MAG,성분부사|시간부사,F,이제,*,*,*,*
정상	NNG,*,T,정상,*,*,*,*
적	XSN,*,T,적,*,*,*,*
으로	JKB,*,F,으로,*,*,*,*
됩니다	VV+EF,*,F,됩니다,Inflect,VV,EF,되/VV/*+ᄇ니다/EF/*
.	SF,*,*,*,*,*,*,*
EOS



In [16]:
# 1.데이터를 가져왔으니 이제 영어, 한글, 띄어쓰기만 남기고 나머지 특수문자를 제거
# -> 정규식을 이용해서 처리
train_df['document'] = train_df['document'].str.replace(r'[^A-Za-z가-힣ㄱ-ㅎㅏ-ㅣ ]', 
                                                        '', 
                                                        regex=True)

In [19]:
display(train_df.head())

,id,document,label
0,9976970,아 더빙 진짜 짜증나네요 목소리,0
1,3819312,흠포스터보고 초딩영화줄오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 솔직히 재미는 없다평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화스파이더맨에서 늙어보이기만 했던 커스틴 던...,1


In [21]:
# 2. 결측치 5개 제거
train_df = train_df.dropna()

In [22]:
print(train_df.shape)

(149995, 3)


In [33]:
# 3. 불용어(stopword) 제거
# 관사, 전치사, 조사, 접속사 => 의미가 없는 단어를 지칭
# 한국어에는 정말 많은 불용어가 존재하는데 이 중 일부만 제거
# 불용어를 제거하는 함수를 이용해서 처리
from konlpy.tag import Okt, Mecab
from tqdm import tqdm

tqdm.pandas()  # tqdm을 판다스에 부착(판다스의 데이터프레임을 처리할때 프로그래스바 표시)
mecab = Mecab()
# okt = Okt()  메캅이나 오케이티 둘 중 하나 사용하면됨


# 형태소 분석 후 불용어를 제거해서 그 결과를 리턴하는 함수
def word_tokenization(text):
    stop_words = [
    # 조사
    '은', '는', '이', '가', '을', '를', '에', '에서', '에게', '한테', '으로', '로',
    '과', '와', '도', '만', '이나', '보다', '처럼', '까지', '부터', '라도', '마저',

    # 접속사
    '그리고', '그러나', '하지만', '그래서', '그러면', '그런데', '따라서', '혹은', '또는',

    # 의존명사/형식적 명사
    '수', '것', '거', '때', '중', '등', '뿐', '대로', '만큼', '따위',

    # 대명사/지시어
    '나', '너', '우리', '저희', '그', '이', '저', '그것', '이것', '저것', '자기',

    # 일반 동사/보조용언
    '되다', '하다', '있다', '없다', '이다', '아니다', '받다', '주다', '되어다', '같다', '되었다',

    # 감탄사/불필요 표현
    '아', '야', '어', '우와', '헐', '음', '응', '네', '예', '자', '좀', '요', '그냥', '또', '그래',

    # 빈도 높은 불필요 어휘
    '정말', '진짜', '너무', '매우', '아주', '항상', '더', '더욱', '계속', '이미', '이제',

    # 불용 보조 용언/어미 어절
    '것이다', '것이', '때문이다', '있습니다', '없습니다', '하는', '해서', '하였다', '했다', '하고', '하며', '하면서',

    # 기타 (프로젝트 목적 따라 제거 가능)
    '하지만', '그러나', '혹시', '그러면', '그럼', '혹은', '만약', '만일', '또는'
    ]

    return [word for word in mecab.morphs(text) if word not in stop_words]  # 형태소 분석도 하면서 불용어도 제거하는 함수를 만들었다

data_train = train_df['document'].progress_apply(word_tokenization)  # apply() 가 함수적용해주는 함수인데 .progress_apply이렇게하면 그 함수를 적용할때 프로그레스바가 보이게 해주는 함수
print(data_train.head())

100%|████████████████████████████████████████████████████████████████| 149995/149995 [00:06<00:00, 21940.68it/s]

0                                     [빙, 짜증, 네요, 목소리]
1    [흠, 포스터, 보고, 초딩, 영화, 줄, 오버, 연기, 조차, 가볍, 지, 않, 구나]
2                                  [재, 밓었다그래서보는것을추천한다]
3                [교도소, 이야기, 구먼, 솔직히, 재미, 없, 다, 평점, 조정]
4    [사이몬페그, 의, 익살, 스런, 연기, 돋보였, 던, 영화, 스파이더맨, 늙, 보...
Name: document, dtype: object


In [ ]:
# 불용어를 잘 처리해줘야된다. 더빙을 더를 지우기도하고 그럼

In [27]:
# 3. 불용어(stopword) 제거
# 관사, 전치사, 조사, 접속사 => 의미가 없는 단어를 지칭
# 한국어에는 정말 많은 불용어가 존재하는데 이 중 일부만 제거
# 불용어를 제거하는 함수를 이용해서 처리
from konlpy.tag import Okt, Mecab
from tqdm import tqdm

tqdm.pandas()  # tqdm을 판다스에 부착(판다스의 데이터프레임을 처리할때 프로그래스바 표시)
# mecab = Mecab()
okt = Okt()  # 메캅이나 오케이티 둘 중 하나 사용하면됨


# 형태소 분석 후 불용어를 제거해서 그 결과를 리턴하는 함수
def word_tokenization(text):
    stop_words = [
    # 조사
    '은', '는', '이', '가', '을', '를', '에', '에서', '에게', '한테', '으로', '로',
    '과', '와', '도', '만', '이나', '보다', '처럼', '까지', '부터', '라도', '마저',

    # 접속사
    '그리고', '그러나', '하지만', '그래서', '그러면', '그런데', '따라서', '혹은', '또는',

    # 의존명사/형식적 명사
    '수', '것', '거', '때', '중', '등', '뿐', '대로', '만큼', '따위',

    # 대명사/지시어
    '나', '너', '우리', '저희', '그', '이', '저', '그것', '이것', '저것', '자기',

    # 일반 동사/보조용언
    '되다', '하다', '있다', '없다', '이다', '아니다', '받다', '주다', '되어다', '같다', '되었다',

    # 감탄사/불필요 표현
    '아', '야', '어', '우와', '헐', '음', '응', '네', '예', '자', '좀', '요', '그냥', '또', '그래',

    # 빈도 높은 불필요 어휘
    '정말', '진짜', '너무', '매우', '아주', '항상', '더', '더욱', '계속', '이미', '이제',

    # 불용 보조 용언/어미 어절
    '것이다', '것이', '때문이다', '있습니다', '없습니다', '하는', '해서', '하였다', '했다', '하고', '하며', '하면서',

    # 기타 (프로젝트 목적 따라 제거 가능)
    '하지만', '그러나', '혹시', '그러면', '그럼', '혹은', '만약', '만일', '또는'
    ]

    return [word for word in okt.morphs(text) if word not in stop_words]  # 형태소 분석도 하면서 불용어도 제거하는 함수를 만들었다

train_df['document'].progress_apply(word_tokenization)  # apply() 가 함수적용해주는 함수인데 .progress_apply이렇게하면 그 함수를 적용할때 프로그레스바가 보이게 해주는 함수

100%|██████████████████████████████████████████████████████████████████| 149995/149995 [06:06<00:00, 409.02it/s]


0                                          [더빙, 짜증나네요, 목소리]
1             [흠, 포스터, 보고, 초딩, 영화, 줄, 오버, 연기, 조차, 가볍지, 않구나]
2                          [무재, 밓었, 다그, 래서, 보는것을, 추천, 한, 다]
3                           [교도소, 이야기, 구먼, 솔직히, 재미, 평점, 조정]
4         [사이, 몬페, 의, 익살스런, 연기, 돋보였던, 영화, 스파이더맨, 늙어, 보이기...
                                ...                        
149995                                [인간, 문제, 지, 소, 뭔, 죄인]
149996                                            [평점, 낮아서]
149997                [게, 뭐, 한국인, 거들, 먹거리, 고, 필리핀, 혼혈, 착하다]
149998             [청춘, 영화, 의, 최고봉, 방황, 우울했던, 날, 들, 의, 자화상]
149999                         [한국, 영화, 최초, 수간, 내용, 담긴, 영화]
Name: document, Length: 149995, dtype: object

In [34]:
# 지금까지 진행하면.. 형태소 분석과 불용어 처리까지 진행할 수 있어요
# 이걸 이용해서 단어사전을 만들 수 있어요
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts(data_train) # 분리되어 있는 형태소가 결국 Token이 되고 이 Token들을 이용해서 vacabulary 생성
# 총 단어 개수 
print(len(tokenizer.word_index))  # 52185

52185


In [38]:
# 이 오만개의 단어를 다 사용해서 문자열을 숫자로 변경할 껀가요?
# 일반적으로 많은 빈도를 가지는 token만을 이용해서 문자열을 숫자 시퀀스로 변경 그럼 몇개가 적당할까요
def get_voca_size(threshold):
    cnt=0
    for x in tokenizer.word_counts.values():
        if x > threshold:
            cnt = cnt+1
    return cnt

voca_size = get_voca_size(5)  # 빈도가 5보다 큰 단어들의 갯수를 알려준다
print(voca_size)

# 단어사전을 생성할 거에요
tokenizer = Tokenizer(oov_token='<OOV>',
                     num_words=15000)

tokenizer.fit_on_texts(data_train)

# 단어 사전을 만들엇으니 이걸 이용해서 문자열을 숫자의 시퀸스로 변경할 수 있어요
data_train_seq = tokenizer.texts_to_sequences(data_train)
print(data_train_seq[0:3])

# 최종적으로 길이만 똑같이 만들어주면 우리 모델에 집어넣을수있는 형태가 완성될 것같아요
from tensorflow.keras.preprocessing.sequence import pad_sequences
x_data_train = pad_sequences(data_train_seq,
                            maxlen=75)
y_data_train = np.asarray(train_df['label'])

# 확인
print(x_data_train[0:1])

13728
[[879, 187, 21, 663], [925, 449, 296, 593, 2, 89, 1512, 34, 754, 912, 11, 27, 331], [162, 1]]
[[  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 879
  187  21 663]]


In [40]:
# 데이터는 준비가 끝났으니 모델만 만들어주면되요
# 데이터 만들때 원핫 인코딩을 하지 않았어요 우린 임베딩을 사용할거에요
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, LSTM
from tensorflow.keras.optimaers import Adam
from tensorflow.keras.callbacks import EarlyStopping, Modelcheckpoint, TensorBoard

model = Sequential()

model.add(Embedding(input_dim=15001,  
                   output_dim=128,
                   input_length=75)) # 패딩으로 토큰수를 몇으로 잡았는지
model.add(LSTM(units=16,
              activation='tanh'))
model.add(Dense(units=1,
               activation='sigmoid'))
model.compile(optimizer=Adam(leraning_rate=1e-3),
             loss='binary_crossentropy',
             metrics=['accuracy'])
# 3가지 콜백을 사용
ex_cb = EarlyStopping(monitor='val_loss',
                     patience=4,
                     retore_best_weights=True,
                     verbose=1)

ModuleNotFoundError: No module named 'tensorflow.keras.optimaers'

In [41]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, LSTM
from tensorflow.keras.optimizers import Adam  
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, TensorBoard 

# 모델 구성
model = Sequential()
model.add(Embedding(input_dim=15001,  # num_words=15000 + oov_token='<OOV>' → 15001
                    output_dim=128,
                    input_length=75))  # 패딩 길이

model.add(LSTM(units=16, activation='tanh'))
model.add(Dense(units=1, activation='sigmoid'))

# 모델 컴파일
model.compile(optimizer=Adam(learning_rate=1e-3), 
              loss='binary_crossentropy',
              metrics=['accuracy'])

# 콜백 정의
ex_cb = EarlyStopping(monitor='val_loss',
                      patience=4,
                      restore_best_weights=True, 
                      verbose=1)


/home/govlept1004/anaconda3/envs/tf_env/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2025-07-09 12:51:19.981537: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-07-09 12:51:20.102092: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-07-09 12:51:20.102157: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-07-09 12:51:20.105856: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not op

In [45]:
# 학습 진행
model.fit(x_data_train,
          y_data_train,
          epochs=100,
          batch_size=64,
          validation_split=0.2,
          callbacks=[ex_cb], 
          verbose=1)


Epoch 1/100


2025-07-09 12:53:46.329828: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8900


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 23s 11ms/step - accuracy: 0.7966 - loss: 0.4333 - val_accuracy: 0.8487 - val_loss: 0.3421
Epoch 2/100
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 22s 12ms/step - accuracy: 0.8780 - loss: 0.2847 - val_accuracy: 0.8566 - val_loss: 0.3344
Epoch 3/100
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 22s 12ms/step - accuracy: 0.8998 - loss: 0.2384 - val_accuracy: 0.8550 - val_loss: 0.3484
Epoch 4/100
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 21s 11ms/step - accuracy: 0.9199 - loss: 0.1955 - val_accuracy: 0.8483 - val_loss: 0.3769
Epoch 5/100
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 22s 12ms/step - accuracy: 0.9339 - loss: 0.1661 - val_accuracy: 0.8464 - val_loss: 0.4255
Epoch 6/100
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 21s 11ms/step - accuracy: 0.9449 - loss: 0.1406 - val_accuracy: 0.8454 - val_loss: 0.4790
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 2.


In [51]:
# 학습이 끝난 다음
# 예측 작업을 수행할 수 있어요
review_sentences = ['내가 만들어도 이것보다는 잘 만들겠다',
                   '너무너무 재미있었어요. 감사합니다.',
                   '아.. 내돈.. 돈이 너무 아까워요!',
                   '이것도 영화라고 쯧쯧... 스토리가 산으로 가요!',
                   '감동과 재미가 있는 영화입니다.',
                   '너무너무 재미없다.. 잠와 죽는줄',
                   '너무너무 재미있다.. 잠이 확깨요']

# 이 리뷰를 모델에 넣어서 예측을 수행
# 데이터 전처리를 해야 해요!

df = pd.DataFrame({'document': review_sentences })
display(df)

# 1. 정규식 이용해서 특수문자 제거(한글, 영어, 공백 제외하고)
df['document'] = df['document'].str.replace(r'[^A-Za-z가-힣ㄱ-ㅎㅏ-ㅣ ]', 
                                            '', 
                                            regex=True)
# 2. 결측치 처리 (없음)
# 3. 형태소 분석하고 불용어를 제거
data_predict = df['document'].progress_apply(word_tokenization)
print(data_predict)

# 4. 이미 만들어놓은 단어사전을 이용해서 우리 문장을 숫자로 변환
data_predict_seq = tokenizer.texts_to_sequences(data_predict)
print(data_predict_seq)

# 5. 숫자로 변환이 됐으니 길이를 맞춰요 75로
x_data_predict = pad_sequences(data_predict_seq,
                              maxlen=75)
result = model.predict(x_data_predict)
print(result)

,document
0,내가 만들어도 이것보다는 잘 만들겠다
1,너무너무 재미있었어요. 감사합니다.
2,아.. 내돈.. 돈이 너무 아까워요!
3,이것도 영화라고 쯧쯧... 스토리가 산으로 가요!
4,감동과 재미가 있는 영화입니다.
5,너무너무 재미없다.. 잠와 죽는줄
6,너무너무 재미있다.. 잠이 확깨요


100%|██████████████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 11613.97it/s]

0       [내, 만들, 어도, 잘, 만들, 겠, 다]
1    [너무너무, 재미있, 었, 어요, 감사, 합니다]
2               [내, 돈, 돈, 아까, 워]
3       [영화, 라고, 쯧쯧, 스토리, 산, 가요]
4           [감동, 재미, 있, 영화, 입니다]
5        [너무너무, 재미없, 다, 잠, 죽, 줄]
6        [너무너무, 재미있, 다, 잠, 확, 깨]
Name: document, dtype: object
[[38, 63, 389, 35, 63, 45, 3], [702, 69, 15, 42, 543, 148], [38, 117, 117, 1116, 1870], [2, 80, 3156, 47, 784, 1623], [56, 67, 12, 2, 100], [702, 72, 3, 657, 172, 89], [702, 69, 3, 657, 1422, 1322]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
[[0.409711  ]
 [0.9919958 ]
 [0.00270161]
 [0.00671416]
 [0.98692024]
 [0.01932739]
 [0.720228  ]]
